# 05 — Dinâmica Temporal do Corpus (análise nova)

Evolução temporal dos regimes iconocráticos e do score ENDURECIMENTO ao longo dos séculos XVI–XX.
Análise descritiva/ilustrativa — não inferencial (viés de seleção, densidade desigual por década).

**Fonte:** `data/processed/corpus_dataset.csv` (165 itens, 158 com ano)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv('../data/processed/corpus_dataset.csv')
df_year = df.dropna(subset=['year']).copy()
df_year['year'] = df_year['year'].astype(int)
df_year['decade'] = (df_year['year'] // 10) * 10

# Focus on 1850s–1970s for statistical density
df_modern = df_year[df_year['decade'] >= 1850].copy()

print(f"Total items: {len(df)}")
print(f"Items with year: {len(df_year)}")
print(f"Modern period (≥1850): {len(df_modern)} items in {df_modern['decade'].nunique()} decades")
print(f"Regime distribution: {df_modern['regime_iconocratico'].value_counts().to_dict()}")

## 5.1 Distribuição de regimes por década (1850s–1970s)

In [ ]:
regime_colors = {
    'fundacional': '#4C72B0',
    'normativo': '#DD8452',
    'militar': '#C44E52',
    'contra-alegoria': '#55A868',
}

ct = pd.crosstab(df_modern['decade'], df_modern['regime_iconocratico'])
# Ensure all regimes present
for r in regime_colors:
    if r not in ct.columns:
        ct[r] = 0
ct = ct[regime_colors.keys()]

fig, ax = plt.subplots(figsize=(14, 6))
ct.plot(kind='bar', stacked=True, ax=ax, color=[regime_colors[c] for c in ct.columns], edgecolor='white', linewidth=0.5)
ax.set_xlabel('Década')
ax.set_ylabel('Número de itens')
ax.set_title('Distribuição de Regimes Iconocráticos por Década (1850s–1970s)')
ax.legend(title='Regime', bbox_to_anchor=(1.02, 1), loc='upper left')

# Annotate 1910s militar spike
if 1910 in ct.index:
    militar_1910 = ct.loc[1910, 'militar']
    ax.annotate(f'WWI\n{militar_1910} itens militar',
                xy=(list(ct.index).index(1910), ct.loc[1910].sum()),
                xytext=(list(ct.index).index(1910) + 1.5, ct.loc[1910].sum() + 2),
                arrowprops=dict(arrowstyle='->', color='black'),
                fontsize=9, ha='center')

plt.tight_layout()
plt.savefig('../data/processed/fig_06_regime_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

# Print decade summary
print("\nItens por década:")
for decade in sorted(ct.index):
    total = ct.loc[decade].sum()
    regimes = ', '.join(f"{r}: {v}" for r, v in ct.loc[decade].items() if v > 0)
    print(f"  {decade}s: {total} ({regimes})")

## 5.2 Tendência do ENDURECIMENTO por década

In [ ]:
# ENDURECIMENTO mean per decade (only decades with >=3 items for error bars)
decade_stats = df_modern.groupby('decade')['purificacao_composto'].agg(['mean', 'std', 'count'])
decade_stats.columns = ['mean_e', 'std_e', 'count']

fig, ax1 = plt.subplots(figsize=(14, 6))

# Bar for item count (background)
ax1.bar(decade_stats.index, decade_stats['count'], width=8, alpha=0.25, color='gray', label='N itens')
ax1.set_xlabel('Década')
ax1.set_ylabel('Número de itens', color='gray')
ax1.tick_params(axis='y', labelcolor='gray')

# Line for ENDURECIMENTO mean
ax2 = ax1.twinx()
ax2.plot(decade_stats.index, decade_stats['mean_e'], 'o-', color='#C44E52', linewidth=2, markersize=8, label='ENDURECIMENTO médio')
# Error bars only for decades with >= 3 items
mask = decade_stats['count'] >= 3
if mask.any():
    ax2.errorbar(decade_stats.index[mask], decade_stats['mean_e'][mask],
                 yerr=decade_stats['std_e'][mask], fmt='none', ecolor='#C44E52', alpha=0.5, capsize=4)
ax2.set_ylabel('Score ENDURECIMENTO (médio)', color='#C44E52')
ax2.tick_params(axis='y', labelcolor='#C44E52')
ax2.set_ylim(0, 3.5)

ax1.set_title('ENDURECIMENTO Médio por Década (1850s–1970s)')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout()
plt.savefig('../data/processed/fig_07_endurecimento_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print("ENDURECIMENTO por década (N ≥ 3):")
for decade in decade_stats.index:
    row = decade_stats.loc[decade]
    flag = ' ⚠' if row['count'] < 3 else ''
    print(f"  {decade}s: mean={row['mean_e']:.2f} ± {row['std_e']:.2f} (n={int(row['count'])}){flag}")

## 5.3 Presença de países por década

In [ ]:
# Which countries appear in which decades?
country_decade = pd.crosstab(df_modern['decade'], df_modern['country'])

# Filter to countries with >= 2 items total
country_totals = country_decade.sum()
major_countries = country_totals[country_totals >= 2].index.tolist()
country_decade_f = country_decade[major_countries]

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(country_decade_f.T, cmap='YlOrRd', linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Número de itens'}, annot=True, fmt='d')
ax.set_xlabel('Década')
ax.set_ylabel('País')
ax.set_title('Presença de Países no Corpus por Década (1850s–1970s)')
plt.tight_layout()
plt.savefig('../data/processed/fig_08_country_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

## 5.4 Distribuição de suportes por década

In [ ]:
medium_decade = pd.crosstab(df_modern['decade'], df_modern['medium_norm'])

fig, ax = plt.subplots(figsize=(14, 6))
medium_decade.plot(kind='bar', stacked=True, ax=ax, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Década')
ax.set_ylabel('Número de itens')
ax.set_title('Distribuição de Suportes (medium_norm) por Década')
ax.legend(title='Suporte', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../data/processed/fig_09_medium_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("Suportes no período moderno:")
print(df_modern['medium_norm'].value_counts().to_string())

## 5.5 Síntese

### Achados principais:
- O pico dos anos 1910 (regime militar, WWI) é o evento mais visível no corpus
- O regime `fundacional` é o mais antigo e persistente; `normativo` emerge ~1650s
- O ENDURECIMENTO médio por década deve ser interpretado com cautela (viés de seleção)

### Limitações:
- Corpus não é amostra aleatória — reflete disponibilidade em arquivos digitais
- Décadas pré-1850 têm densidade insuficiente para análise estatística
- Tendências temporais podem refletir produção de selos (século XX) vs. pinturas (séculos anteriores)

### Fontes historiográficas para contextualização:
- Sbriccoli: criminal law codification waves in 19th-c. Europe
- Broedel: allegory hardening in bureaucratic contexts
- Mondzain (2002): image economy and state iconography